In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
performance_features = [
    "return_measurement_comparison_percent",
    "return_lag_1q",
    "return_lag_2q",
    "return_lag_3q",
    "return_lag_4q",
    "return_change_1q",
    "return_trailing_4q",
    "return_mean_4q",
    "return_std_4q",
    "return_min_4q",
    "positive_quarter_share_4q",
    "current_return_minus_peer_median",
    "current_return_peer_percentile",
]

volatility_features = [
    "return_investment_five_year_volatility_comparison_percent",
    "return_investment_ten_year_volatility_comparison_percent_clean",
    "volatility_5y_minus_10y",
    "volatility_5y_missing",
    "volatility_10y_missing",
]

saa_features = [
    "strategic_growth_allocation",
    "allocation_equity",
    "allocation_property",
    "allocation_infrastructure",
    "allocation_cash",
    "allocation_alternatives",
    "allocation_credit",
    "allocation_fixed_income_excluding_credit",
    "saa_missing",
]

hedging_features = [
    "weighted_currency_hedging_ratio",
    "currency_hedging_applicable_allocation",
    "currency_hedging_distinct_ratios",
]

strategy_change_features = [
    "strategic_growth_change_1q",
    "equity_allocation_change_1q",
    "cash_allocation_change_1q",
]

baseline_features = (
    performance_features
    + volatility_features
    + saa_features
    + hedging_features
    + strategy_change_features
)

In [3]:
# imprt the data
DATA_PATH = Path(
    "../data/processed/hist_model_mysuper_features.parquet"
)

hist_model_mysuper_features = pd.read_parquet(
    DATA_PATH
)

hist_model_mysuper_features.shape

(13719, 136)

In [4]:
# ensure that the dat is in datetime format
hist_model_mysuper_features[
    "period_end_date"
] = pd.to_datetime(
    hist_model_mysuper_features[
        "period_end_date"
    ]
)

hist_model_mysuper_features[
    "target_end_date"
] = pd.to_datetime(
    hist_model_mysuper_features[
        "target_end_date"
    ]
)

In [5]:
# check coluns
missing_features = [
    col
    for col in baseline_features
    if col not in hist_model_mysuper_features.columns
]

missing_features

[]

### Create the modelling population

In [6]:
target_col = "future_4q_bottom_quartile"

model_data = (
    hist_model_mysuper_features[
        hist_model_mysuper_features[
            "future_4q_target_available_option"
        ].eq(True)
        &
        hist_model_mysuper_features[
            target_col
        ].notna()
    ]
    .copy()
)

# convert target from boolean to 1/0 system
model_data[target_col] = (
    model_data[target_col]
    .astype("int8")
)

# sort the table
entity_keys = [
    "rse_abn",
    "abn_product_identifier",
    "abn_investment_menu_identifier",
    "abn_investment_option_identifier",
]

model_data = (
    model_data
    .sort_values(
        ["period_end_date"]
        + entity_keys
    )
    .reset_index(drop=True)
)

In [7]:
model_data.shape

(11720, 136)

In [8]:
model_data[target_col].value_counts(
    normalize=True
)

future_4q_bottom_quartile
0    0.741553
1    0.258447
Name: proportion, dtype: float64

### Inspect the temporal distribution

In [9]:
# define peer fields
peer_keys = [
    "rse_abn",
    "abn_investment_option_identifier",
    "period_end_date",
]

option_target_nunique = (
    model_data
    .groupby(peer_keys)[target_col]
    .nunique()
)

option_target_nunique.value_counts()

future_4q_bottom_quartile
1    9896
Name: count, dtype: int64

In [10]:
option_target_nunique.max()

np.int64(1)

In [11]:
option_quarter_target = (
    model_data
    .groupby(
        peer_keys,
        as_index=False
    )
    .agg(
        future_4q_bottom_quartile=(
            target_col,
            "first",
        )
    )
)

# quarterly summary
row_summary = (
    model_data
    .groupby("period_end_date")
    .agg(
        modelling_rows=(
            target_col,
            "size",
        ),
        row_positive_rate=(
            target_col,
            "mean",
        ),
        target_end_min=(
            "target_end_date",
            "min",
        ),
        target_end_max=(
            "target_end_date",
            "max",
        ),
    )
)

option_summary = (
    option_quarter_target
    .groupby("period_end_date")
    .agg(
        unique_options=(
            "abn_investment_option_identifier",
            "size",
        ),
        option_positive_rate=(
            target_col,
            "mean",
        ),
    )
)

quarter_summary = (
    row_summary
    .join(option_summary)
)

quarter_summary

,modelling_rows,row_positive_rate,target_end_min,target_end_max,unique_options,option_positive_rate
period_end_date,,,,,,
2014-12-31,185,0.254054,2015-12-31,2015-12-31,153,0.254902
2015-03-31,185,0.318919,2016-03-31,2016-03-31,153,0.254902
2015-06-30,185,0.329730,2016-06-30,2016-06-30,153,0.254902
2015-09-30,185,0.308108,2016-09-30,2016-09-30,153,0.254902
2015-12-31,185,0.302703,2016-12-31,2016-12-31,153,0.254902
2016-03-31,185,0.259459,2017-03-31,2017-03-31,153,0.254902
2016-06-30,186,0.258065,2017-06-30,2017-06-30,154,0.253247
2016-09-30,187,0.262032,2017-09-30,2017-09-30,155,0.258065
2016-12-31,200,0.260000,2017-12-31,2017-12-31,168,0.250000


The temporal inspection looks good enough to proceed, and it supports the split strategy we chose. A few things are especially important.

First, the target timing is internally consistent. For every quarter, target_end_min == target_end_max, and it is exactly four quarters after period_end_date. For example, 2023-03-31 → 2024-03-31 and 2024-06-30 → 2025-06-30.

Because of that, safely continue to the next part.

### Prepare the final test period

For the first baseline, the last 4 available quarters will be considered as the final untouched test set.

In [12]:
# define the available quarters
available_quarters = (
    model_data[
        "period_end_date"
    ]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

len(available_quarters)

42

In [13]:
TEST_QUARTERS = 4

test_quarters = (
    available_quarters[
        -TEST_QUARTERS:
    ]
)

test_start = test_quarters[0]
test_end = test_quarters[-1]

test_start, test_end

(Timestamp('2024-06-30 00:00:00'), Timestamp('2025-03-31 00:00:00'))

### Define data used for development

In [14]:
# define data used for development based on final test start date cutoff
development_mask = (
    model_data[
        "target_end_date"
    ]
    <
    test_start
)

development_data = (
    model_data[
        development_mask
    ]
    .copy()
)

# define the final test data
final_test_data = (
    model_data[
        model_data[
            "period_end_date"
        ].isin(test_quarters)
    ]
    .copy()
)

In [15]:
development_data.shape, final_test_data.shape

((9003, 136), (1369, 136))

### Identify the label-maturation gap

identify rows whose quarters are between the last development prediction quarter and the first test quarter, which simply haven't matured by the simulated test date.

In [16]:
maturation_gap = (
    model_data[
        (model_data["period_end_date"] < test_start)
        &
        (model_data["target_end_date"] >= test_start)
    ]
    .copy()
)

# inspect result
maturation_gap[
    [
        "period_end_date",
        "target_end_date",
    ]
].drop_duplicates().sort_values(
    "period_end_date"
)

,period_end_date,target_end_date
9003,2023-06-30,2024-06-30
9348,2023-09-30,2024-09-30
9682,2023-12-31,2024-12-31
10017,2024-03-31,2025-03-31


### Build the expanding validation folds

In [17]:
# define number of folds and number of quarters inside each validation
N_VALIDATION_FOLDS = 4
VALIDATION_QUARTERS = 4

development_quarters = (
    development_data[
        "period_end_date"
    ]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

len(development_quarters)

34

In [18]:
required_validation_quarters = (
    N_VALIDATION_FOLDS
    * VALIDATION_QUARTERS
)

# Check that there is enough history
assert (
    len(development_quarters)
    >
    required_validation_quarters
)

In [19]:
# Take the latest 16 development quarters as the validation windows
validation_quarters_all = (
    development_quarters[-required_validation_quarters:]
)

# Create 4 blocks
validation_blocks = []

for i in range(N_VALIDATION_FOLDS):
    start = (i * VALIDATION_QUARTERS)
    end = (start + VALIDATION_QUARTERS)

    validation_blocks.append(
        validation_quarters_all[start:end]
    )

In [20]:
# inspect result
for i, block in enumerate(
    validation_blocks,
    start=1,
):
    print(
        f"Fold {i}:",
        block[0],
        "to",
        block[-1],
    )

Fold 1: 2019-06-30 00:00:00 to 2020-03-31 00:00:00
Fold 2: 2020-06-30 00:00:00 to 2021-03-31 00:00:00
Fold 3: 2021-06-30 00:00:00 to 2022-03-31 00:00:00
Fold 4: 2022-06-30 00:00:00 to 2023-03-31 00:00:00


### apply fold-specific purging

In [21]:
folds = []

for fold_number, val_quarters in enumerate(
    validation_blocks,
    start=1,
):

    validation_start = (val_quarters[0])

    validation_end = (val_quarters[-1])

    train_mask = (
        (development_data["period_end_date"] < validation_start)
        &
        (development_data["target_end_date"] < validation_start)
    )

    validation_mask = (
        development_data["period_end_date"].isin(
            val_quarters
        )
    )

    train_data = (
        development_data[train_mask].copy()
    )

    validation_data = (
        development_data[validation_mask].copy()
    )

    folds.append(
        {
            "fold": fold_number,
            "validation_start": validation_start,
            "validation_end": validation_end,
            "train_index": train_data.index,
            "validation_index": validation_data.index,
        }
    )

In [22]:
# audit result
fold_summary = []

for fold in folds:

    train_data = (
        development_data.loc[
            fold["train_index"]
        ]
    )

    validation_data = (
        development_data.loc[
            fold["validation_index"]
        ]
    )

    fold_summary.append(
        {
            "fold":
                fold["fold"],

            "train_start":
                train_data[
                    "period_end_date"
                ].min(),

            "train_end":
                train_data[
                    "period_end_date"
                ].max(),

            "train_target_end_max":
                train_data[
                    "target_end_date"
                ].max(),

            "train_rows":
                len(train_data),

            "train_positive_rate":
                train_data[
                    target_col
                ].mean(),

            "validation_start":
                validation_data[
                    "period_end_date"
                ].min(),

            "validation_end":
                validation_data[
                    "period_end_date"
                ].max(),

            "validation_rows":
                len(validation_data),

            "validation_positive_rate":
                validation_data[
                    target_col
                ].mean(),
        }
    )

fold_summary = pd.DataFrame(
    fold_summary
)

fold_summary

,fold,train_start,train_end,train_target_end_max,train_rows,train_positive_rate,validation_start,validation_end,validation_rows,validation_positive_rate
0,1,2014-12-31,2018-03-31,2019-03-31,2722,0.272594,2019-06-30,2020-03-31,1259,0.249404
1,2,2014-12-31,2019-03-31,2020-03-31,3783,0.265662,2020-06-30,2021-03-31,1387,0.268205
2,3,2014-12-31,2020-03-31,2021-03-31,5042,0.261603,2021-06-30,2022-03-31,1254,0.270335
3,4,2014-12-31,2021-03-31,2022-03-31,6429,0.263027,2022-06-30,2023-03-31,1320,0.250000


In [23]:
(
    fold_summary[
        "train_target_end_max"
    ]
    <
    fold_summary[
        "validation_start"
    ]
).all()

np.True_

In [24]:
# inspect dataypes and missingness
feature_audit = pd.DataFrame(
    {
        "dtype": model_data[
            baseline_features
        ].dtypes.astype(str),

        "missing_count": model_data[
            baseline_features
        ].isna().sum(),

        "missing_rate": model_data[
            baseline_features
        ].isna().mean(),
    }
).sort_values(
    "missing_rate",
    ascending=False,
)

feature_audit

,dtype,missing_count,missing_rate
volatility_5y_minus_10y,float64,8725,0.744454
return_investment_ten_year_volatility_comparison_percent_clean,float64,8719,0.743942
return_investment_five_year_volatility_comparison_percent,float64,6016,0.513311
strategic_growth_change_1q,float64,3577,0.305205
cash_allocation_change_1q,float64,3577,0.305205
equity_allocation_change_1q,float64,3577,0.305205
currency_hedging_applicable_allocation,float64,3217,0.274488
allocation_fixed_income_excluding_credit,float64,3217,0.274488
currency_hedging_distinct_ratios,Int64,3217,0.274488
allocation_infrastructure,float64,3217,0.274488


In [25]:
non_numeric_features = [
    col
    for col in baseline_features
    if not pd.api.types.is_numeric_dtype(
        model_data[col]
    )
    and not pd.api.types.is_bool_dtype(
        model_data[col]
    )
]

non_numeric_features

[]

In [26]:
numeric_features = (
    model_data[
        baseline_features
    ]
    .select_dtypes(
        include=["number"]
    )
    .columns
)

infinite_counts = (
    np.isinf(
        model_data[
            numeric_features
        ]
    )
    .sum()
)

infinite_counts[
    infinite_counts > 0
]

Series([], dtype: Int64)

### Setting up Pipeline

In [27]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression